# 3. Build the organisational-context features

This notebook links the technical user-days to monthly organisational records and constructs the three context representations used in Environment A.

**Inputs:** `features_cyberlog.parquet` from Notebook 2 and `ldap_monthly.parquet` from Notebook 1. Outcome and insider-indicator columns are removed when the technical table is loaded.

**Outputs:** `context_rawcat.parquet` for M1, `context_curated.parquet` for M2, `context_shuffled.parquet` for M3, `split_definition.json` and the fitted context profiles.

The curated Business Intelligence (BI) context represents role, department, an access proxy, expected hours and peer behaviour. Frequency encodings, hours profiles and peer statistics are fitted on training data and then held fixed.

The date boundaries are constants here, but their design used the previously documented distribution of malicious activity. The partition design is therefore not label blind. The context output tables contain keys and features, not labels.

*Documentation copy, 8 November 2026. Notes edited for clarity; code, saved outputs and execution metadata are unchanged. This copy has not been rerun.*


In [1]:
# ============================================================
# 0. Setup — load inputs, drop labels immediately
# ============================================================
import os, glob, json, random, warnings
import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED); np.random.seed(SEED)
warnings.filterwarnings("ignore")
WORK = "/kaggle/working"

f_hits = glob.glob("/kaggle/input/**/features_cyberlog.parquet", recursive=True)
l_hits = glob.glob("/kaggle/input/**/ldap_monthly.parquet", recursive=True)
assert f_hits, "Notebook 2 output not attached (Add Input -> Your Work -> NB2)"
assert l_hits, "Notebook 1 output not attached (Add Input -> Your Work -> NB1)"
print("features:", f_hits[0])
print("ldap    :", l_hits[0])

feats = pd.read_parquet(f_hits[0])
DROPPED = [c for c in ("label", "user_is_insider") if c in feats.columns]
feats = feats.drop(columns=DROPPED)
print(f"LEAKAGE CONTROL: dropped label columns on load: {DROPPED}")
if not DROPPED:
    print("NOTE: no 'label'/'user_is_insider' columns found to drop. That is safe here —\n"
          "      contexts are built from an explicit feature whitelist and never carry\n"
          "      labels — but update the cell-6 audit text if NB2 names its label column\n"
          "      differently and you want the printed line to reflect the real name.")

# Feature columns this notebook depends on (define once; reused in Section 3 + shuffled control).
HOUR_COLS  = ["first_logon_hour", "last_logoff_hour"]
BASE_FEATS = ["logon_count", "logon_afterhours", "logon_pcs",
              "device_connects", "file_events", "file_to_removable",
              "emails_sent", "email_external", "email_size",
              "http_count", "http_afterhours", "http_distinct_domains"]

# Fail fast (and legibly) if NB2's output uses different names, instead of a deep KeyError later.
missing = [c for c in HOUR_COLS + BASE_FEATS if c not in feats.columns]
assert not missing, (
    "features_cyberlog.parquet is missing required columns: "
    f"{missing}\nAvailable columns: {sorted(feats.columns)}\n"
    "Edit HOUR_COLS / BASE_FEATS above to match Notebook 2's column names.")
print(f"required feature columns present: {len(HOUR_COLS + BASE_FEATS)} OK")

feats["user"] = feats["user"].astype(str)
feats["day"]  = pd.to_datetime(feats["day"])
feats["month"] = feats["day"].dt.to_period("M").astype(str)

ldap = pd.read_parquet(l_hits[0])
ldap["user_id"] = ldap["user_id"].astype(str)
print("feats:", feats.shape, "| ldap:", ldap.shape)

features: /kaggle/input/notebooks/thandondhlovu10/2-feature-engineering/features_cyberlog.parquet
ldap    : /kaggle/input/notebooks/thandondhlovu10/1-cert-r6-2-eda-dataset-preparation/ldap_monthly.parquet
LEAKAGE CONTROL: dropped label columns on load: ['label', 'user_is_insider']
required feature columns present: 14 OK
feats: (1394010, 44) | ldap: (68923, 11)


## 1. Apply the recorded chronological split

The saved windows are January to July 2010 for training, August to December 2010 for validation, and January to May 2011 for test. Observations outside these windows are marked for exclusion from modelling.

The dates were chosen with knowledge of the scenario timing: training has no identified malicious days, validation contains scenarios 1, 3 and 5, and test contains scenarios 2 and 4. This cell does not read labels; it writes the agreed constants to `split_definition.json` for the modelling notebooks.


In [2]:
# ============================================================
# 1. Split definition — frozen
# ============================================================
SPLIT = {
    "train_start": "2010-01-01", "train_end": "2010-07-31",
    "val_start":   "2010-08-01", "val_end":   "2010-12-31",
    "test_start":  "2011-01-01", "test_end":  "2011-05-31",
    "rationale": ("Train window verified free of malicious activity; validation "
                  "holds scenarios 1/3/5, test holds scenarios 2/4 (different "
                  "attack types than validation). Frozen before any modelling."),
    "seed": SEED,
}
with open(f"{WORK}/split_definition.json", "w") as fh:
    json.dump(SPLIT, fh, indent=2)

# Every window is bounded on BOTH sides; anything outside all three -> "drop"
# (e.g. the stray 2011-06-01 logon day must not fall into test past test_end).
d = feats["day"]
feats["partition"] = np.select(
    [(d >= SPLIT["train_start"]) & (d <= SPLIT["train_end"]),
     (d >= SPLIT["val_start"])   & (d <= SPLIT["val_end"]),
     (d >= SPLIT["test_start"])  & (d <= SPLIT["test_end"])],
    ["train", "val", "test"], default="drop")
print(feats["partition"].value_counts())
n_drop = int((feats["partition"] == "drop").sum())
print(f"user-days outside all frozen windows (dropped): {n_drop}")
TRAIN = feats["partition"] == "train"

partition
train    592378
val      406070
test     395549
drop         13
Name: count, dtype: int64
user-days outside all frozen windows (dropped): 13


## 2. Join the monthly organisational records

Match each user-day to its user-month LDAP record. Missing intermediate attributes are carried forward within the user; remaining earlier gaps are backfilled from the first available record.

The checks require complete role coverage and more than one real role category. They test the join, not whether a later record was valid for earlier activity. The original audit does not quantify which observations, if any, required backfilling, so historical validity remains a separate qualification.


In [3]:
# ============================================================
# 2. LDAP slowly-changing-dimension join
# ============================================================
ATTRS = ["role", "functional_unit", "department", "team", "supervisor"]
snap = (ldap[["user_id", "snapshot"] + ATTRS]
        .rename(columns={"user_id": "user", "snapshot": "month"}))
snap["month"] = snap["month"].astype(str)   # 'YYYY-MM' str, or Period -> 'YYYY-MM'

um = feats[["user", "month"]].drop_duplicates()
um = um.merge(snap, on=["user", "month"], how="left")
um = um.sort_values(["user", "month"])
um[ATTRS] = um.groupby("user")[ATTRS].ffill()
um[ATTRS] = um.groupby("user")[ATTRS].bfill()   # users active before 1st snapshot appearance

# Hard sanity gate: if the join key silently mismatched (e.g. ldap['snapshot'] is not
# 'YYYY-MM' but a full date), every row goes NaN and ffill/bfill cannot recover -> M2
# would be built on all-UNKNOWN attributes while the run still "succeeds". Halt instead.
unmatched = int(um["role"].isna().sum())
print(f"user-months without any LDAP attributes: {unmatched}/{len(um)} (expect 0)")
assert unmatched == 0, (
    f"{unmatched}/{len(um)} user-months got no LDAP attributes after ffill/bfill.\n"
    "Likely a join-key format mismatch. Check that ldap['snapshot'] is 'YYYY-MM'.\n"
    f"  snapshot sample: {sorted(snap['month'].unique())[:3]}\n"
    f"  feats month sample: {sorted(feats['month'].unique())[:3]}")
um[ATTRS] = um[ATTRS].fillna("UNKNOWN")

ctx = feats[["user", "day", "month", "partition"]].merge(
        um, on=["user", "month"], how="left")
print("ctx:", ctx.shape)
print("\nrole inventory (train users per role):")
role_inv = (ctx.loc[TRAIN.values, ["user", "role"]].drop_duplicates()
              .groupby("role").size().sort_values(ascending=False))
print(role_inv.to_string())

# Guard against a join that passed the count check but yielded no real roles.
n_real_roles = int((role_inv.index != "UNKNOWN").sum())
assert n_real_roles > 1, (
    f"Only {n_real_roles} non-UNKNOWN role(s) among train users — the LDAP join looks "
    "degenerate (expected dozens of roles). Inspect the snapshot join before continuing.")

user-months without any LDAP attributes: 0/65278 (expect 0)
ctx: (1394010, 9)

role inventory (train users per role):
role
ProductionLineWorker        413
ElectricalEngineer          406
Technician                  401
Salesman                    358
MechanicalEngineer          295
StockroomClerk              280
MaterialsEngineer           117
Scientist                   113
HardwareEngineer            112
SystemsEngineer             111
ComputerScientist           106
Mathematician               106
SecurityGuard               105
ITAdmin                     100
Physicist                    94
SoftwareDeveloper            93
ComputerProgrammer           91
AdministrativeAssistant      84
Manager                      80
TestEngineer                 78
WebDeveloper                 74
ChiefEngineer                72
TechnicalWriter              57
SoftwareQualityEngineer      42
SoftwareEngineer             38
Supervisor                   28
NursePractitioner            27
Director     

## 3. Construct the 18 curated features for M2

The five contextual dimensions are represented by four feature families:

| Family | Features | Construction |
|---|---:|---|
| Access proxy | 1 | A three-tier rule based on role keywords. |
| Role and department frequencies | 2 | Shares of distinct training users in each category. |
| Expected hours | 3 | Early-start and late-end deviations, plus an outside-hours indicator. |
| Peer behaviour | 12 | Deviations from training-fitted behavioural references. |

Expected hours use the role's 10th-percentile first-logon time and 90th-percentile last-logoff time, with an organisation-wide fallback for insufficient support. Peer references use role and department, with the role/global fallbacks implemented below. Profiles remain fixed after training. Team and supervisor are excluded from M2.

The recorded rule amendment dated 4 July 2026 changed the highest-tier keyword from `admin` to `itadmin`, because `admin` also matched clerical administrative roles. The code retains that rule. The tier is a proxy, not an audited permission record.

The next two cells define the construction functions, build M2's context block and save its fitted profiles.


In [4]:
# ============================================================
# 3a. Derivation functions (reused verbatim for the shuffled control)
# ============================================================
# BASE_FEATS and HOUR_COLS are defined once in the setup cell (Section 0).

TIER3_KW = ["itadmin"]   # v2: was ["admin"], which wrongly matched clerical
                          # AdministrativeAssistant/Staff (see markdown above)
TIER2_KW = ["manager","director","president","executive","chief",
            "supervisor","auditor","attorney","scientist"]

def access_level(role_series):
    r = role_series.astype(str).str.lower()
    lvl = pd.Series(1, index=role_series.index, dtype="int32")
    lvl[r.apply(lambda s: any(k in s for k in TIER2_KW))] = 2
    lvl[r.apply(lambda s: any(k in s for k in TIER3_KW))] = 3
    return lvl

def fit_freq(train_users_attr, col):
    # share of distinct TRAIN users per category
    t = train_users_attr[["user", col]].drop_duplicates()
    return (t.groupby(col)["user"].nunique() / t["user"].nunique()).to_dict()

def fit_expected_hours(df_train, min_users=15):
    ok = df_train[df_train["first_logon_hour"] >= 0]
    g_users = ok.groupby("role")["user"].nunique()
    prof = ok.groupby("role").agg(
        exp_start=("first_logon_hour", lambda s: np.percentile(s, 10)),
        exp_end=("last_logoff_hour",   lambda s: np.percentile(s[s >= 0], 90)
                                                  if (s >= 0).any() else np.nan))
    glob_start = np.percentile(ok["first_logon_hour"], 10)
    glob_end   = np.percentile(ok.loc[ok["last_logoff_hour"] >= 0,
                                      "last_logoff_hour"], 90)
    small = g_users[g_users < min_users].index
    prof.loc[prof.index.isin(small), ["exp_start","exp_end"]] = [glob_start, glob_end]
    prof = prof.fillna({"exp_start": glob_start, "exp_end": glob_end})
    return prof.reset_index(), glob_start, glob_end

def fit_peer_stats(df_train, min_users=15):
    df = df_train.copy()
    df["grp"] = df["role"].astype(str) + "|" + df["department"].astype(str)
    users_per_grp = df.groupby("grp")["user"].nunique()
    small = users_per_grp[users_per_grp < min_users].index
    df.loc[df["grp"].isin(small), "grp"] = df.loc[df["grp"].isin(small), "role"]
    stats = df.groupby("grp")[BASE_FEATS].agg(["mean", "std"])
    stats.columns = [f"{a}__{b}" for a, b in stats.columns]
    glob = {f"{c}__mean": df[c].mean() for c in BASE_FEATS}
    glob.update({f"{c}__std": df[c].std() for c in BASE_FEATS})
    return stats.reset_index(), glob, set(small)

def assign_group(df, small_groups):
    grp = df["role"].astype(str) + "|" + df["department"].astype(str)
    grp[grp.isin(small_groups)] = df.loc[grp.isin(small_groups), "role"].astype(str)
    return grp

def derive_context(ctx_attrs, feats, tag=""):
    """ctx_attrs: user-day frame with role/department (+partition). Fits on train,
    applies frozen everywhere. Returns (context_df, profiles_dict)."""
    df = ctx_attrs.merge(
        feats[["user","day","first_logon_hour","last_logoff_hour"] + BASE_FEATS],
        on=["user","day"], how="left")
    tr = df[df["partition"] == "train"]

    out = df[["user","day"]].copy()
    out["access_level"] = access_level(df["role"])

    role_freq = fit_freq(tr, "role"); dept_freq = fit_freq(tr, "department")
    out["role_freq"] = df["role"].map(role_freq).fillna(min(role_freq.values()))
    out["dept_freq"] = df["department"].map(dept_freq).fillna(min(dept_freq.values()))

    hours, gs, ge = fit_expected_hours(tr)
    df = df.merge(hours, on="role", how="left")
    df["exp_start"] = df["exp_start"].fillna(gs)
    df["exp_end"]   = df["exp_end"].fillna(ge)
    has = df["first_logon_hour"] >= 0
    out["early_start_dev"] = np.where(has,
        (df["exp_start"] - df["first_logon_hour"]).clip(lower=0), 0)
    out["late_end_dev"] = np.where(df["last_logoff_hour"] >= 0,
        (df["last_logoff_hour"] - df["exp_end"]).clip(lower=0), 0)
    out["outside_expected_hours"] = ((out["early_start_dev"] > 0) |
                                     (out["late_end_dev"] > 0)).astype("int32")

    stats, glob, small = fit_peer_stats(tr)
    df["grp"] = assign_group(df, small)
    df = df.merge(stats, on="grp", how="left")
    for c in BASE_FEATS:
        m = df[f"{c}__mean"].fillna(glob[f"{c}__mean"])
        s = df[f"{c}__std"].fillna(glob[f"{c}__std"]).replace(0, np.nan)
        out[f"{c}_peer_z"] = ((df[c] - m) / s).replace(
            [np.inf, -np.inf], np.nan).fillna(0)

    profiles = {"role_freq": role_freq, "dept_freq": dept_freq,
                "hours": hours, "peer_stats": stats, "small_groups": list(small)}
    print(f"[{tag}] context features: {out.shape[1]-2} cols x {len(out):,} rows")
    return out, profiles

In [5]:
# ============================================================
# 3b. Build curated context (M2) + persist fitted profiles
# ============================================================
ctx_cur, prof = derive_context(ctx, feats, tag="curated")
ctx_cur.to_parquet(f"{WORK}/context_curated.parquet", index=False)

# appendix artefacts
rule_tbl = (ctx.loc[TRAIN.values, ["role"]].drop_duplicates()
              .assign(access_level=lambda d: access_level(d["role"]))
              .sort_values(["access_level","role"], ascending=[False, True]))
rule_tbl.to_csv(f"{WORK}/access_level_rules.csv", index=False)
prof["hours"].to_parquet(f"{WORK}/expected_hours_profiles.parquet", index=False)
prof["peer_stats"].to_parquet(f"{WORK}/peer_group_profiles.parquet", index=False)

print("\naccess-level distribution (train users):")
print(rule_tbl.groupby("access_level").size())
print("\nexpected-hours profile sample:")
print(prof["hours"].head(8).to_string(index=False))


[curated] context features: 18 cols x 1,394,010 rows

access-level distribution (train users):
access_level
1    34
2    11
3     1
dtype: int64

expected-hours profile sample:
                   role  exp_start  exp_end
             Accountant        7.0     19.0
AdministrativeAssistant        7.0     19.0
    AdministrativeStaff        7.0     19.0
               Attorney        7.0     19.0
          ChiefEngineer        7.0     19.0
     ComputerProgrammer        7.0     18.0
      ComputerScientist        7.0     18.0
               Director        7.0     19.0


## 4. Construct the raw organisational comparator for M1

Encode role, functional unit and department using indicator columns for categories observed in training. Team and supervisor use training-user frequency encodings.

An unseen indicator category receives zeros in that block; an unseen team or supervisor receives frequency zero. These differ from M2's minimum-frequency fallback for unseen role or department categories.

M1 tests a different representation, not simply a larger copy of M2. Its feature count differs from the curated configuration.


In [6]:
# ============================================================
# 4. Raw categorical comparator
# ============================================================
raw = ctx[["user","day"]].copy()
for col in ("role","functional_unit","department"):
    train_cats = ctx.loc[TRAIN.values, col].unique()
    oh = pd.get_dummies(
        pd.Categorical(ctx[col], categories=train_cats), dtype="int8")
    oh.columns = [f"{col}__{str(c)[:40]}" for c in oh.columns]
    raw = pd.concat([raw, oh.set_index(raw.index)], axis=1)
for col in ("team","supervisor"):
    fr = fit_freq(ctx[TRAIN.values], col)
    raw[f"{col}_freq"] = ctx[col].map(fr).fillna(0)

# Truncating category names can, in principle, collide two categories into one column.
# Fail loudly if it happens (raise the [:40] limit in this cell if so).
dupes = raw.columns[raw.columns.duplicated()].tolist()
assert not dupes, (f"Duplicate one-hot column names after truncation: {dupes[:5]} ... "
                   "raise the [:40] truncation length in this cell.")

raw.to_parquet(f"{WORK}/context_rawcat.parquet", index=False)
print("raw categorical comparator:", raw.shape)

raw categorical comparator: (1394010, 84)


## 5. Construct the shuffled-context control for M3

A seed-42 user permutation reassigns the latest available role and department records. Each user's reassigned attributes remain static across the observation period. The same context functions then rebuild access, frequency, hours and peer features using the shuffled assignments and training behaviour.

The technical activity itself is not shuffled. M3 therefore matches M2's feature count but not its monthly organisational history, and its derived hours and peer features still contain behavioural information.

A supported improvement by M3 would be a warning to investigate the comparison. It would not, by itself, prove leakage. Conversely, a non-triggered warning would not prove that all contextual dependencies had been removed.


In [7]:
# ============================================================
# 5. Shuffled negative control
# ============================================================
rng = np.random.default_rng(SEED)
users = ctx["user"].unique()
perm  = dict(zip(users, rng.permutation(users)))

user_attrs = (ctx.sort_values("month")
                 .groupby("user")[["role","functional_unit","department",
                                   "team","supervisor"]].last())
shuf_attrs = user_attrs.loc[[perm[u] for u in users]].reset_index(drop=True)
shuf_attrs.index = users

ctx_shuf = ctx[["user","day","month","partition"]].copy()
for c in ("role","department"):
    ctx_shuf[c] = ctx_shuf["user"].map(shuf_attrs[c])

ctx_s, _ = derive_context(ctx_shuf, feats, tag="shuffled")
ctx_s.to_parquet(f"{WORK}/context_shuffled.parquet", index=False)


[shuffled] context features: 18 cols x 1,394,010 rows


## 6. Check the context outputs

The audit reports the columns dropped on load, the training-fitted profiles and the saved split. It checks that the three context tables contain no label or insider columns, then lists the output files.

These are checks of the declared construction rules. They do not resolve the LDAP-backfill qualification or prove that the shuffled control removes every source of contextual information.


In [8]:
# ============================================================
# 6. Leakage audit + outputs
# ============================================================
print("LEAKAGE AUDIT")
print("-" * 60)
print(f"1. Label columns dropped on load:            {DROPPED}")
print( "2. Answer key files read in this notebook:   NONE (no insiders/")
print( "   malicious_events path appears in this notebook's code).")
print(f"3. All fitted objects use partition=='train' only:")
print(f"   - frequency encodings (role/dept/team/supervisor)")
print(f"   - expected-hours profiles ({len(prof['hours'])} roles)")
print(f"   - peer-group stats ({len(prof['peer_stats'])} groups; "
      f"{len(prof['small_groups'])} small groups fell back to role)")
print(f"4. Split frozen in split_definition.json:    {SPLIT['train_start']} .. "
      f"{SPLIT['test_end']}")
print(f"5. Context outputs carry NO label columns:")
for f in ("context_curated","context_rawcat","context_shuffled"):
    cols = pd.read_parquet(f"{WORK}/{f}.parquet").columns
    assert not any("label" in c or "insider" in c for c in cols)
    print(f"   - {f}: OK ({len(cols)} cols)")
print("-" * 60)
print("\nfiles in /kaggle/working:")
for f in sorted(os.listdir(WORK)):
    p = f"{WORK}/{f}"
    if os.path.isfile(p):
        print(f"  {f:38s} {os.path.getsize(p)/1e6:8,.1f} MB")


LEAKAGE AUDIT
------------------------------------------------------------
1. Label columns dropped on load:            ['label', 'user_is_insider']
2. Answer key files read in this notebook:   NONE (no insiders/
   malicious_events path appears in this notebook's code).
3. All fitted objects use partition=='train' only:
   - frequency encodings (role/dept/team/supervisor)
   - expected-hours profiles (46 roles)
   - peer-group stats (56 groups; 72 small groups fell back to role)
4. Split frozen in split_definition.json:    2010-01-01 .. 2011-05-31
5. Context outputs carry NO label columns:
   - context_curated: OK (20 cols)
   - context_rawcat: OK (84 cols)
   - context_shuffled: OK (20 cols)
------------------------------------------------------------

files in /kaggle/working:
  __notebook__.ipynb                          0.0 MB
  access_level_rules.csv                      0.0 MB
  context_curated.parquet                    18.9 MB
  context_rawcat.parquet                      0.9 